In [ ]:
pip install pandas deepl

In [3]:
import pandas
import deepl
import re

# Introducimos la clave de Deepl y asignamos la variable "translator".
AUTH_KEY = "your_deepl_api_key_here"
translator = deepl.Translator(AUTH_KEY)

# Cargamos el glosario en xlxs para que lo procese.
df_glosario = pd.read_excel("glosario.xlsx")

# Definimos la función para utilizarla más adelante.
def traducir_magic(texto_original):
    # Ordenamos el glosario, primero por prioridad, y, si dos prioridades coinciden, toma la longitud de texto más larga.
    df_glosario['longitud'] = df_glosario['term_en'].str.len()
    glosario_ordenado = df_glosario.sort_values(
        by=['prioridad', 'longitud'],
        ascending=[False, False]
    )

    traducciones_temporales = {}
    texto_preparado = texto_original
    # Una vez ordenado, tomamos las filas que necesitaremos más adelante
    for i, fila in glosario_ordenado.iterrows():
        en = str(fila['term_en'])
        es = str(fila['term_esp'])

        # Usamos regex para encontrar palabras completas y evitar errores como "csobretrol"
        patron = re.compile(r'\b' + re.escape(en) + r'\b', re.IGNORECASE)

        if patron.search(texto_preparado):
            marcador = f"[[TEMP_{i}]]"
            texto_preparado = patron.sub(marcador, texto_preparado)
            traducciones_temporales[marcador] = es

    traducciones_temporales = {}
    texto_preparado = texto_original

    # Una vez ordenado, tomamos las filas que necesitaremos más adelante.
    for i, fila in glosario_ordenado.iterrows():
        en = str(fila['term_en'])
        es = str(fila['term_esp'])

        # Ponemos varias condiciones para la identificación de términos:
        # que sea insensible a las mayúsculas.
        # que solo detecte las palabras completas (evitar el problema csobretrol).
        if en.lower() in texto_preparado.lower():
            patron = re.compile(r'\b' + re.escape(en) + r'\b', re.IGNORECASE)

            # Solo si cumple las condiciones (patron), se sustituyen por marcadores [[...]], para que sean ilegibles para Deepl y evitar el problema cementrenio.
            if patron.search(texto_preparado):
                marcador = f"[[TEMP_{i}]]"
                texto_preparado = patron.sub(marcador, texto_preparado)
                traducciones_temporales[marcador] = es

    # Traducimos lo restante con Deepl, que evitará los términos entre corchetes (marcadores).
    resultado_deepl = translator.translate_text(
        texto_preparado,
        target_lang="ES"
    ).text

    # Volvemos a sustituir los marcadores codificados [[...]] por el término correspondiente.
    texto_final = resultado_deepl
    for marcador, palabra_es in traducciones_temporales.items():
        texto_final = texto_final.replace(marcador, palabra_es)

    return texto_final

ModuleNotFoundError: No module named 'deepl'

In [ ]:
# El usuario introduce el texto para traducir y se lleva a cabo toda la función,antes definida.
if __name__ == "__main__":
    texto_usuario = input("Introduzca el texto de reglas a traducir: ")
    resultado = traducir_magic(texto_usuario)
    print(f"Resultado: {resultado}")